# BIT 4133 Natural Language Processing
## Week 12: End-to-End Speech Processing Models

### Practical Focus
Build a simple End-to-End Speech Recognition System that:

- Accepts live speech input
- Converts speech into text
- Displays the recognized text
- Extracts audio features
- Tests at least five different speakers
- Evaluates recognition performance
- Discusses limitations and possible improvements


In [ ]:
# Install the libraries required for speech recognition, audio processing, and evaluation
!pip install -q transformers librosa soundfile jiwer


In [ ]:
# Import standard Python libraries
import os
import subprocess

# Import deep learning and audio processing libraries
import torch
import librosa
import librosa.display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import the Hugging Face pipeline for automatic speech recognition
from transformers import pipeline

# Import Word Error Rate for evaluating speech recognition performance
from jiwer import wer

# Import tools for audio playback, JavaScript, and microphone recording in Google Colab
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Confirm that all libraries were imported successfully
print("All libraries imported successfully.")


## Automatic Speech Recognition

Automatic Speech Recognition (ASR) is the process of converting spoken language into written text.

### Traditional Speech Recognition
Traditional systems use several separate components such as:

1. Audio feature extraction
2. Acoustic model
3. Pronunciation model
4. Language model
5. Text output

### End-to-End Speech Recognition
End-to-end models learn to convert audio directly into text using a single deep learning architecture.

In this practical, a pretrained Whisper model is used as the end-to-end speech recognition model.


In [ ]:
# Check whether a GPU is available
# Use GPU device 0 when available; otherwise use the CPU
device = 0 if torch.cuda.is_available() else -1

print("Loading speech recognition model...")

# Load the pretrained Whisper Tiny English model
# The model converts recorded speech directly into text
asr_model = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny.en",
    device=device
)

# Confirm that the model has loaded
print("Speech recognition model loaded successfully.")
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")


In [ ]:
# JavaScript code used to access the browser microphone in Google Colab
RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));

const blobToDataURL = blob => new Promise(resolve => {
    const reader = new FileReader();
    reader.onloadend = () => resolve(reader.result);
    reader.readAsDataURL(blob);
});

async function recordAudio(time) {
    // Ask the user for microphone access
    const stream = await navigator.mediaDevices.getUserMedia({audio: true});

    // Create a browser audio recorder
    const recorder = new MediaRecorder(stream);
    const chunks = [];

    // Store each recorded audio chunk
    recorder.ondataavailable = event => chunks.push(event.data);

    // Start recording
    recorder.start();

    // Record for the specified amount of time
    await sleep(time);

    // Stop recording
    recorder.stop();

    // Wait until the recording fully stops
    await new Promise(resolve => {
        recorder.onstop = resolve;
    });

    // Turn off the microphone
    stream.getTracks().forEach(track => track.stop());

    // Combine the recorded chunks into one audio file
    const blob = new Blob(chunks);

    // Convert the audio file into a format Python can receive
    return await blobToDataURL(blob);
}
"""

# Define a Python function for recording audio from the microphone
def record_audio(filename="recorded_audio.webm", seconds=6):
    # Load the JavaScript recording code into the notebook
    display(Javascript(RECORD_JS))

    print(f"Recording for {seconds} seconds. Speak now...")

    # Run the JavaScript recording function
    audio_data = output.eval_js(
        f"recordAudio({seconds * 1000})"
    )

    # Decode the recorded audio data
    audio_binary = b64decode(audio_data.split(",")[1])

    # Save the recorded audio to a file
    with open(filename, "wb") as file:
        file.write(audio_binary)

    print("Recording completed.")

    # Return the name of the saved file
    return filename


In [ ]:
# Record a 6-second live speech sample using the computer microphone
webm_file = record_audio(
    filename="live_speech.webm",
    seconds=6
)

# Set the filename for the converted WAV audio
wav_file = "live_speech.wav"

# Convert the browser-recorded WebM file into WAV format using FFmpeg
subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i",
        webm_file,
        wav_file
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Confirm that the audio file was saved
print("Audio saved as:", wav_file)

# Display an audio player so the recorded speech can be played back
display(Audio(wav_file))


In [ ]:
# Load the recorded WAV audio file
# sr=None keeps the original sample rate of the recording
audio, sample_rate = librosa.load(
    wav_file,
    sr=None
)

# Display basic information about the audio file
print("Audio loaded successfully.")
print("Sample Rate:", sample_rate)
print("Number of Audio Samples:", len(audio))
print("Duration:", round(len(audio) / sample_rate, 2), "seconds")


In [ ]:
# Create a figure for the speech waveform
plt.figure(figsize=(10, 4))

# Plot the waveform of the recorded audio
librosa.display.waveshow(
    audio,
    sr=sample_rate
)

# Add labels and a title to the graph
plt.title("Speech Waveform")
plt.xlabel("Time")
plt.ylabel("Amplitude")
plt.grid(True)

# Display the waveform
plt.show()


In [ ]:
# Extract 13 MFCC features from the recorded speech
# MFCCs represent important characteristics of the speech signal
mfcc = librosa.feature.mfcc(
    y=audio,
    sr=sample_rate,
    n_mfcc=13
)

# Display the size of the extracted MFCC feature matrix
print("MFCC Feature Shape:", mfcc.shape)

# Create a figure for the MFCC feature visualization
plt.figure(figsize=(10, 5))

# Display the MFCC features as a heatmap
librosa.display.specshow(
    mfcc,
    sr=sample_rate,
    x_axis="time"
)

# Add a color scale, title, and axis labels
plt.colorbar()
plt.title("MFCC Features of Recorded Speech")
plt.xlabel("Time")
plt.ylabel("MFCC Coefficients")

# Display the MFCC plot
plt.show()


In [ ]:
# Pass the recorded audio file to the Whisper speech recognition model
result = asr_model(wav_file)

# Extract the recognized text from the model output
recognized_text = result["text"].strip()

# Display the speech recognition result clearly
print("=" * 60)
print("SPEECH RECOGNITION RESULT")
print("=" * 60)
print("Recognized Text:")
print(recognized_text)


In [ ]:
# Create an empty list to store results from all five speakers
speaker_results = []

# Define a reusable function for recording and evaluating each speaker
def test_speaker(
    speaker_number,
    speaker_name,
    reference_text,
    seconds=6
):
    # Display the current speaker being tested
    print("=" * 60)
    print(f"TESTING SPEAKER {speaker_number}: {speaker_name}")
    print("=" * 60)

    # Create unique filenames for each speaker
    webm_filename = f"speaker_{speaker_number}.webm"
    wav_filename = f"speaker_{speaker_number}.wav"

    # Record the speaker's voice
    record_audio(
        filename=webm_filename,
        seconds=seconds
    )

    # Convert the recorded WebM audio into WAV format
    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-i",
            webm_filename,
            wav_filename
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    # Convert the recorded speech into text using the Whisper model
    prediction = asr_model(
        wav_filename
    )["text"].strip()

    # Calculate the Word Error Rate by comparing expected and recognized text
    error_rate = wer(
        reference_text.lower(),
        prediction.lower()
    )

    # Convert Word Error Rate into an approximate recognition accuracy percentage
    accuracy = max(
        0,
        (1 - error_rate) * 100
    )

    # Store the speaker's evaluation results
    speaker_results.append({
        "Speaker": speaker_name,
        "Reference Text": reference_text,
        "Recognized Text": prediction,
        "WER": round(error_rate, 4),
        "Recognition Accuracy (%)": round(accuracy, 2)
    })

    # Display the expected sentence, recognized sentence, and accuracy
    print("\nExpected Text:")
    print(reference_text)

    print("\nRecognized Text:")
    print(prediction)

    print(
        "\nRecognition Accuracy:",
        round(accuracy, 2),
        "%"
    )


In [ ]:
# Test Speaker 1 using the first reference sentence
# Use a real different speaker for the final assignment
test_speaker(
    speaker_number=1,
    speaker_name="Speaker 1",
    reference_text="Natural language processing allows computers to understand human language."
)


In [ ]:
# Test Speaker 2 using the second reference sentence
# Use a real different speaker for the final assignment
test_speaker(
    speaker_number=2,
    speaker_name="Speaker 2",
    reference_text="Artificial intelligence is changing the way people interact with technology."
)


In [ ]:
# Test Speaker 3 using the third reference sentence
# Use a real different speaker for the final assignment
test_speaker(
    speaker_number=3,
    speaker_name="Speaker 3",
    reference_text="Speech recognition converts spoken language into written text."
)


In [ ]:
# Test Speaker 4 using the fourth reference sentence
# Use a real different speaker for the final assignment
test_speaker(
    speaker_number=4,
    speaker_name="Speaker 4",
    reference_text="Machine learning models can learn patterns from large amounts of data."
)


In [ ]:
# Test Speaker 5 using the fifth reference sentence
# Use a real different speaker for the final assignment
test_speaker(
    speaker_number=5,
    speaker_name="Speaker 5",
    reference_text="End to end speech recognition uses deep learning to process audio."
)


In [ ]:
# Convert the list of speaker results into a pandas DataFrame
results_df = pd.DataFrame(
    speaker_results
)

# Display the complete evaluation table
print("SPEECH RECOGNITION EVALUATION RESULTS")
display(results_df)


In [ ]:
# Calculate the average recognition accuracy across all tested speakers
average_accuracy = results_df[
    "Recognition Accuracy (%)"
].mean()

# Calculate the average Word Error Rate across all tested speakers
average_wer = results_df[
    "WER"
].mean()

# Display the overall performance of the speech recognition system
print("=" * 60)
print("OVERALL SPEECH RECOGNITION PERFORMANCE")
print("=" * 60)

print(
    "Average Recognition Accuracy:",
    round(average_accuracy, 2),
    "%"
)

print(
    "Average Word Error Rate:",
    round(average_wer, 4)
)


In [ ]:
# Create a bar chart comparing recognition accuracy for the five speakers
plt.figure(figsize=(9, 5))

# Plot each speaker's recognition accuracy
plt.bar(
    results_df["Speaker"],
    results_df["Recognition Accuracy (%)"]
)

# Add the chart title and axis labels
plt.title(
    "Speech Recognition Accuracy for Five Speakers"
)

plt.xlabel("Speaker")
plt.ylabel("Recognition Accuracy (%)")

# Keep the accuracy scale between 0% and 100%
plt.ylim(0, 100)

# Add horizontal grid lines to make comparison easier
plt.grid(axis="y")

# Display the chart
plt.show()


## Limitations

The speech recognition system may be affected by background noise, microphone quality, speaking speed, pronunciation, accents, and unclear speech. The small Whisper model may also produce less accurate results compared to larger speech recognition models.

## Possible Improvements

The system can be improved by using higher-quality microphones, reducing background noise, collecting more diverse speech samples, testing larger speech recognition models, applying audio noise reduction, and training or fine-tuning the model using speech data from different accents and speakers.


## Student Reflection

This week, I learned how end-to-end speech recognition systems convert spoken language into text using deep learning. I recorded speech, extracted waveform and MFCC features, tested a Whisper model, and evaluated recognition accuracy across different speakers. I also understood how noise, accents, and microphone quality affect speech recognition performance.
